# 3. Feature Engineering

*Proposal section 3.2.2.* Turns cleaned transactions into one row per user, splits users into synthetic workers and clients, and derives the trust features and labels each model uses. Saves the worker and client datasets for `04_model_training`.

## Setup

Runs locally or in Google Colab. In Colab it mounts Google Drive, copies this repo's code from GitHub, and points `data/` at **My Drive** (raw files in the Drive root, outputs in `My Drive/processed`) and `models/` at `My Drive/models`, so outputs survive between sessions.

In [1]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_URL = "https://github.com/JamieNguru/169421-escrow-based-wallet-and-payment-system.git"
    if not Path("/content/repo").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "dev", REPO_URL, "/content/repo"], check=True)
    ROOT = Path("/content/repo/trust-scoring-ml")
    DRIVE = Path("/content/drive/MyDrive")
    (DRIVE / "models").mkdir(exist_ok=True)
    for name, target in [("data", DRIVE), ("models", DRIVE / "models")]:
        if not (ROOT / name).exists():
            (ROOT / name).symlink_to(target, target_is_directory=True)
else:
    ROOT = Path.cwd().parent

sys.path.insert(0, str(ROOT))
DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
import pandas as pd

from src.feature_engineering import (
    add_client_rate_features,
    add_client_trust_label,
    add_time_features,
    add_worker_rate_features,
    add_worker_trust_label,
    aggregate_client_behavior,
    assign_synthetic_roles,
    build_client_dataset,
    build_worker_dataset,
)
from src.train import CLIENT_FEATURE_COLUMNS, WORKER_FEATURE_COLUMNS

In [3]:
transactions = pd.read_parquet(PROCESSED_DIR / "transactions_clean.parquet")
print(f"{len(transactions):,} cleaned transactions")

13,305,915 cleaned transactions


## Time features

In [4]:
transactions = add_time_features(transactions)
transactions[["date", "hour", "day_of_week", "month", "is_weekend"]].head()

,date,hour,day_of_week,month,is_weekend
0,2010-01-01 00:01:00,0,4,1,0
1,2010-01-01 00:02:00,0,4,1,0
2,2010-01-01 00:02:00,0,4,1,0
3,2010-01-01 00:05:00,0,4,1,0
4,2010-01-01 00:06:00,0,4,1,0


## One row per user

Each user's transactions are aggregated into volume, amount, failure count and rate, and the average gap between transactions.

In [5]:
client_agg = aggregate_client_behavior(transactions)
del transactions
print(f"{len(client_agg):,} users")
client_agg.describe().round(3)

1,219 users


,client_id,transaction_count,total_amount,avg_amount,error_count,avg_hours_between_transactions,error_rate
count,1219.000,1219.000,1219.000,1219.000,1219.000,1219.000,1219.000
mean,1004.916,10915.435,469102.151,43.909,173.415,9.575,0.016
std,582.384,5607.355,292602.852,17.478,153.994,5.422,0.007
min,0.000,760.000,26605.340,5.340,7.000,1.778,0.002
25%,496.500,7223.500,269693.835,32.224,99.000,6.404,0.012
50%,1019.000,9832.000,398837.160,40.721,137.000,8.599,0.014
75%,1514.500,13349.000,597774.025,52.549,202.000,11.599,0.017
max,1998.000,48479.000,2445773.250,147.244,2935.000,113.484,0.146


## Synthetic workers and clients

The data has no roles, and the proposal's trust features (job completion, disputes, response time for workers; payment completion, escrow release time, refunds for clients) don't exist in it. Each user is randomly assigned a role (fixed seed, so reproducible) and their real behaviour becomes proxy features: failure rate stands in for disputes and refunds, and the gap between transactions for response and release time.

In [6]:
client_agg = assign_synthetic_roles(client_agg, seed=42)
client_agg["role"].value_counts()

role
client    627
worker    592
Name: count, dtype: int64

In [7]:
worker_dataset = build_worker_dataset(client_agg)
client_dataset = build_client_dataset(client_agg)
print("Workers:", worker_dataset.shape, " Clients:", client_dataset.shape)
worker_dataset.head()

Workers: (592, 5)  Clients: (627, 5)


,client_id,job_completion_rate,dispute_count,response_time_hours,total_jobs
0,0,0.985307,188,6.734881,12795
1,3,0.965839,205,14.355542,6001
2,4,0.990627,141,5.728938,15043
3,11,0.987793,122,8.623213,9994
4,14,0.989047,64,14.750745,5843


In [8]:
client_dataset.head()

,client_id,payment_completion_rate,escrow_release_time_hours,refund_requests,total_jobs_paid
0,1,0.984513,8.554834,156,10073
1,2,0.987655,8.120777,131,10612
2,5,0.986393,18.036785,65,4777
3,13,0.990575,12.119356,67,7109
4,17,0.974208,4.863067,457,17719


## Trust labels

There is no ground-truth trust label, so one is derived: completion rate minus the normalised dispute/refund rate and normalised response/release time, split into equal Low / Medium / High thirds. This is a modelling assumption, not real ground truth (see `05_model_evaluation` for the check against real fraud). Labels are shown here; `04_model_training` derives them the same way from the saved datasets.

In [9]:
pd.DataFrame({
    "workers": add_worker_trust_label(worker_dataset)["trust_level"].value_counts(),
    "clients": add_client_trust_label(client_dataset)["trust_level"].value_counts(),
})

,workers,clients
trust_level,,
Low,198,209
Medium,197,209
High,197,209


## Model features

Each model gets three features: a completion rate, a dispute or refund **rate** (count divided by jobs), and a time measure. Raw counts and job volume are left out: the label is built from rates, and feeding volume in made predictions depend on how many jobs a user had done rather than how they behaved (see issue #64).

The features are not scaled. Random Forests split on thresholds, so scaling doesn't change their predictions.

**Note:** the completion rate and the dispute (or refund) rate are both calculated from the same failure rate (completion = 1 - failure rate, dispute rate = failure rate), so they always add up to 1. The tables below show it: same standard deviation, means summing to 1. Each model therefore has two independent inputs, not three, and splits its feature importance evenly between the mirrored pair. Separating them would need a second, distinct failure signal, for example treating insufficient-balance errors as payment failures and the other error types as disputes.

In [10]:
print("Worker features:", WORKER_FEATURE_COLUMNS)
add_worker_rate_features(worker_dataset)[WORKER_FEATURE_COLUMNS].describe().round(4)

Worker features: ['job_completion_rate', 'dispute_rate', 'response_time_hours']


,job_completion_rate,dispute_rate,response_time_hours
count,592.0000,592.0000,592.0000
mean,0.9842,0.0158,9.8209
std,0.0080,0.0080,6.2806
min,0.8536,0.0016,1.7776
25%,0.9824,0.0118,6.4494
50%,0.9858,0.0142,8.6989
75%,0.9882,0.0176,11.7603
max,0.9984,0.1464,113.4843


In [11]:
print("Client features:", CLIENT_FEATURE_COLUMNS)
add_client_rate_features(client_dataset)[CLIENT_FEATURE_COLUMNS].describe().round(4)

Client features: ['payment_completion_rate', 'refund_rate', 'escrow_release_time_hours']


,payment_completion_rate,refund_rate,escrow_release_time_hours
count,627.0000,627.0000,627.0000
mean,0.9845,0.0155,9.3420
std,0.0064,0.0064,4.4560
min,0.9360,0.0030,2.0841
25%,0.9828,0.0118,6.2784
50%,0.9861,0.0139,8.4415
75%,0.9882,0.0172,11.5201
max,0.9970,0.0640,39.6900


## Save

In [12]:
for name, df in [("worker_features.csv", worker_dataset), ("client_features.csv", client_dataset)]:
    df.to_csv(PROCESSED_DIR / name, index=False)
    print(f"Saved {(PROCESSED_DIR / name).relative_to(ROOT).as_posix()}: {df.shape}")

Saved data/processed/worker_features.csv: (592, 5)


Saved data/processed/client_features.csv: (627, 5)
